In [2]:
!pip install segmentation-models-pytorch
!pip install albumentations
!pip install opencv-python
!pip install pyyaml
!pip install matplotlib
!pip install tqdm

In [1]:
import torch
print(torch.cuda.is_available())  # Should print: True
print(torch.cuda.get_device_name(0))  # Should show your GPU name


True
NVIDIA GeForce RTX 4060


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

# Custom Dataset
class CrackSegmentationDataset(Dataset):
    def __init__(self, data_yaml_path, split='train', transform=None):
        with open(data_yaml_path, 'r') as f:
            data = yaml.safe_load(f)

        self.root_dir = Path(data_yaml_path).parent
        self.image_dir = self.root_dir / 'images' / split
        self.label_dir = self.root_dir / 'labels' / split

        self.images = list(self.image_dir.glob('*.jpg')) + list(self.image_dir.glob('*.png'))
        print(f"[INFO] Found {len(self.images)} images in {self.image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def yolo_to_mask(self, label_path, img_shape):
        h, w = img_shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        if not label_path.exists():
            return mask

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 6:
                continue

            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
            coords[:, 0] *= w
            coords[:, 1] *= h
            coords = coords.astype(np.int32)
            cv2.fillPoly(mask, [coords], 1)

        return mask

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label_path = self.label_dir / f"{img_path.stem}.txt"
        mask = self.yolo_to_mask(label_path, image.shape)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.unsqueeze(0).float()

# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, masks in tqdm(dataloader, desc="Training"):
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

# Validation function
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validation"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
    return total_loss / len(dataloader)

# Main training loop
def train_unet_resnet(data_yaml_path, epochs=50, batch_size=4, learning_rate=1e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[INFO] Using device: {device}")

    train_transform = A.Compose([
        A.Resize(640, 640),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

    val_transform = A.Compose([
        A.Resize(640, 640),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

    train_dataset = CrackSegmentationDataset(data_yaml_path, 'train', train_transform)
    val_dataset = CrackSegmentationDataset(data_yaml_path, 'val', val_transform)

    print(f"[DEBUG] Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    model = smp.Unet(
        encoder_name="resnet50",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None
    ).to(device)

    criterion = smp.losses.DiceLoss(mode='binary')
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    best_val_loss = float('inf')
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        print(f"\n[INFO] Epoch {epoch + 1}/{epochs}")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = validate_epoch(model, val_loader, criterion, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_unet_resnet50_crack.pth')
            print("[INFO] 🔥 Saved best model")

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_curve.png")
    plt.show()

    return model

# Predict Function
def predict_mask(model, image_path, device):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    transform = A.Compose([
        A.Resize(640, 640),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

    augmented = transform(image=image)
    img_tensor = augmented['image'].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        mask = torch.sigmoid(output).squeeze().cpu().numpy()
        mask = (mask > 0.5).astype(np.uint8) * 255
    return mask

# === RUN ===
if __name__ == "__main__":
    data_yaml_path = r"data/crack-seg/crack-seg.yaml"

    model = train_unet_resnet(
        data_yaml_path=data_yaml_path,
        epochs=50,
        batch_size=4,
        learning_rate=1e-4
    )

    # Example prediction:
    # predicted_mask = predict_mask(model, r"path\to\image.jpg", device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

<local>/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Using device: cuda
[INFO] Found 3812 images in data/crack-seg/images\train
[INFO] Found 208 images in data/crack-seg/images\val
[DEBUG] Train size: 3812, Val size: 208

[INFO] Epoch 1/50


Validation: 100%|██████████| 52/52 [00:11<00:00,  4.47it/s]


Train Loss: 0.5583 | Val Loss: 0.2898 | LR: 0.000100
[INFO] 🔥 Saved best model

[INFO] Epoch 2/50


Validation: 100%|██████████| 52/52 [00:08<00:00,  6.27it/s]


Train Loss: 0.2838 | Val Loss: 0.2595 | LR: 0.000100
[INFO] 🔥 Saved best model

[INFO] Epoch 3/50


... (92 outputs left out)

Validation: 100%|██████████| 52/52 [00:06<00:00,  7.85it/s]


Train Loss: 0.2020 | Val Loss: 0.2333 | LR: 0.000006

[INFO] Epoch 50/50


Validation: 100%|██████████| 52/52 [00:05<00:00,  8.67it/s]


Train Loss: 0.2018 | Val Loss: 0.2331 | LR: 0.000003


In [2]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from pathlib import Path
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import yaml
from torch.utils.data import Dataset

# -------------------------------
# Dataset Class (same as training)
# -------------------------------
class CrackSegmentationDataset(Dataset):
    def __init__(self, data_yaml_path, split='val', transform=None):
        with open(data_yaml_path, 'r') as f:
            data = yaml.safe_load(f)

        self.root_dir = Path(data_yaml_path).parent
        self.image_dir = self.root_dir / 'images' / split
        self.label_dir = self.root_dir / 'labels' / split

        self.images = list(self.image_dir.glob('*.jpg')) + list(self.image_dir.glob('*.png'))
        print(f"[INFO] Found {len(self.images)} images in {self.image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def yolo_to_mask(self, label_path, img_shape):
        h, w = img_shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        if not label_path.exists():
            return mask

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 6:
                continue

            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
            coords[:, 0] *= w
            coords[:, 1] *= h
            coords = coords.astype(np.int32)
            cv2.fillPoly(mask, [coords], 1)

        return mask

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label_path = self.label_dir / f"{img_path.stem}.txt"
        mask = self.yolo_to_mask(label_path, image.shape)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.unsqueeze(0).float()

# -------------------------------
# Evaluation Function
# -------------------------------
def evaluate_model(model, dataloader, device):
    model.eval()
    iou_scores = []
    dice_scores = []
    pixel_accuracies = []

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Evaluating", leave=True):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs) > 0.5

            for pred, true in zip(preds, masks):
                pred = pred.squeeze().cpu().numpy().astype(bool)
                true = true.squeeze().cpu().numpy().astype(bool)

                intersection = np.logical_and(pred, true).sum()
                union = np.logical_or(pred, true).sum()
                iou = intersection / (union + 1e-7)
                dice = (2 * intersection) / (pred.sum() + true.sum() + 1e-7)
                acc = (pred == true).sum() / pred.size

                iou_scores.append(iou)
                dice_scores.append(dice)
                pixel_accuracies.append(acc)

    print(f"\n[Evaluation Metrics]")
    print(f"Mean IoU:        {np.mean(iou_scores):.4f}")
    print(f"Mean Dice:       {np.mean(dice_scores):.4f}")
    print(f"Pixel Accuracy:  {np.mean(pixel_accuracies):.4f}")

    return {
        "iou": np.mean(iou_scores),
        "dice": np.mean(dice_scores),
        "pixel_acc": np.mean(pixel_accuracies)
    }

# -------------------------------
# Run Evaluation
# -------------------------------
if __name__ == "__main__":
    data_yaml_path = r"data/crack-seg/crack-seg.yaml"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    val_transform = A.Compose([
        A.Resize(640, 640),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

    val_dataset = CrackSegmentationDataset(data_yaml_path, split='val', transform=val_transform)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    if len(val_dataset) == 0:
        print("[ERROR] No validation data found. Check the dataset paths and labels.")
        exit()

    # Initialize model and load weights
    model = smp.Unet(
        encoder_name="resnet50",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None
    )
    model.load_state_dict(torch.load('best_unet_resnet50_crack.pth', map_location=device))
    model.to(device)

    # Run evaluation
    metrics = evaluate_model(model, val_loader, device)


[INFO] Found 208 images in data/crack-seg/images\val


<home>\AppData\Local\Temp\ipykernel_13096\665998602.py:139: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_unet_resnet50_crack.pth', ma


[Evaluation Metrics]
Mean IoU:        0.6303
Mean Dice:       0.7576
Pixel Accuracy:  0.9908


In [5]:
import torch
from sklearn.metrics import confusion_matrix

# Initialize counters
TP = 0
FP = 0
FN = 0
TN = 0

for images, masks in tqdm(val_loader):
    images = images.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.sigmoid(outputs)
        preds = (preds > 0.5).float()

    # Flatten to calculate confusion matrix
    preds_np = preds.cpu().numpy().astype(np.uint8).flatten()
    masks_np = masks.cpu().numpy().astype(np.uint8).flatten()

    # Compute per-batch confusion
    batch_TP = np.sum((preds_np == 1) & (masks_np == 1))
    batch_FP = np.sum((preds_np == 1) & (masks_np == 0))
    batch_FN = np.sum((preds_np == 0) & (masks_np == 1))
    batch_TN = np.sum((preds_np == 0) & (masks_np == 0))

    TP += batch_TP
    FP += batch_FP
    FN += batch_FN
    TN += batch_TN

# Compute precision, recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall + 1e-8)

print("\n[Detailed Crack Detection Metrics]")
print(f"TP (Crack predicted as Crack) = {TP}")
print(f"FN (Crack predicted as Background) = {FN}")
print(f"FP (Background predicted as Crack) = {FP}")
print(f"TN (Background predicted as Background) = {TN}")
print()
print(f"Recall (Crack)     = {recall * 100:.2f}%")
print(f"Precision (Crack)  = {precision * 100:.2f}%")
print(f"F1 Score           = {f1_score:.4f}")


100%|██████████| 208/208 [00:06<00:00, 31.91it/s]


[Detailed Crack Detection Metrics]
TP (Crack predicted as Crack) = 1354063
FN (Crack predicted as Background) = 364674
FP (Background predicted as Crack) = 421254
TN (Background predicted as Background) = 83056809

Recall (Crack)     = 78.78%
Precision (Crack)  = 76.27%
F1 Score           = 0.7751
